<a href="https://colab.research.google.com/github/ntx-lab/clifford-distribution/blob/main/clifford_enumeration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
!{sys.executable} -m pip install qiskit

from collections import deque
from qiskit import QuantumCircuit
from qiskit.quantum_info import Clifford

def clifford_key(cliff: Clifford):
    """Turn a Clifford into a hashable key."""
    # Qiskit Clifford has tableau / symplectic data internally.
    # Converting to dict-like immutable representation for dedup.
    return (
        tuple(map(tuple, cliff.symplectic_matrix)),
        tuple(cliff.phase)
    )

def gate_cliffords(n):
    """Generate the basic Clifford generators on n qubits."""
    gens = []

    # H, S on each qubit
    for q in range(n):
        qc = QuantumCircuit(n)
        qc.h(q)
        gens.append(Clifford(qc))

        qc = QuantumCircuit(n)
        qc.s(q)
        gens.append(Clifford(qc))

    # CX on each ordered pair
    for c in range(n):
        for t in range(n):
            if c != t:
                qc = QuantumCircuit(n)
                qc.cx(c, t)
                gens.append(Clifford(qc))

    return gens

def enumerate_cliffords(n, max_count=None):
    """
    Enumerate the n-qubit Clifford group by BFS from identity.
    Works only for small n (e.g. n=1,2, sometimes 3 depending on patience).
    """
    identity = Clifford(QuantumCircuit(n))
    gens = gate_cliffords(n)

    seen = {}
    q = deque([identity])

    k0 = clifford_key(identity)
    seen[k0] = identity

    while q:
        cur = q.popleft()

        if max_count is not None and len(seen) >= max_count:
            break

        for g in gens:
            nxt = cur.compose(g)
            k = clifford_key(nxt)
            if k not in seen:
                seen[k] = nxt
                q.append(nxt)

    return list(seen.values())

if __name__ == "__main__":
    cliffords_1 = enumerate_cliffords(1)
    print("1-qubit Clifford count =", len(cliffords_1))  # should be 24

    cliffords_2 = enumerate_cliffords(2)
    print("2-qubit Clifford count =", len(cliffords_2))  # should be 11520

    # cliffords_3 = enumerate_cliffords(3)
    # print("3-qubit Clifford count =", len(cliffords_3))  # too long to run

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 1.5 MB/s eta 0:00:00
1-qubit Clifford count = 24
2-qubit Clifford count = 11520
